In [1]:
import sys, time
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, IsolationForest
from sklearn.metrics import average_precision_score, precision_recall_curve

from src.data import load_transactions
from src.train import FEATURES, split_by_time

In [2]:
train_df, test_df = split_by_time(load_transactions())
X_train, y_train = train_df[FEATURES], train_df["Class"]
X_test, y_test = test_df[FEATURES], test_df["Class"]

def recall_at_precision(y, scores, min_p=0.8):
    p, r, _ = precision_recall_curve(y, scores)
    ok = p >= min_p
    return r[ok].max() if ok.any() else 0.0

def single_row_ms(score_fn, X, n=200):
    start = time.perf_counter()
    for i in range(n):
        score_fn(X.iloc[[i]])
    return (time.perf_counter() - start) / n * 1000

In [3]:
models = {
    "LogReg": Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000)),
    ]),
    "RandomForest": RandomForestClassifier(
        n_estimators=100, max_depth=12, class_weight="balanced_subsample",
        n_jobs=-1, random_state=42),
    "HistGradBoost": HistGradientBoostingClassifier(
        class_weight="balanced", random_state=42),
}

rows = []
for name, m in models.items():
    m.fit(X_train, y_train)
    scores = m.predict_proba(X_test)[:, 1]
    rows.append({
        "model": name,
        "pr_auc": average_precision_score(y_test, scores),
        "recall@p80": recall_at_precision(y_test, scores),
        "ms_per_row": single_row_ms(lambda X: m.predict_proba(X), X_test),
    })
pd.DataFrame(rows).round(4)

,model,pr_auc,recall@p80,ms_per_row
0,LogReg,0.7622,0.7467,1.2378
1,RandomForest,0.8114,0.7733,17.6734
2,HistGradBoost,0.7355,0.7200,2.0394


In [4]:
iso = IsolationForest(n_estimators=100, contamination=0.0017, random_state=42, n_jobs=-1)
iso.fit(X_train)
iso_scores = -iso.score_samples(X_test)

print("PR-AUC    :", average_precision_score(y_test, iso_scores))
print("recall@p80:", recall_at_precision(y_test, iso_scores))

PR-AUC    : 0.025510889624925943
recall@p80: 0.0
